In [2]:
import torch
import torchaudio
torch.random.manual_seed(0)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(torch.__version__)
print(torchaudio.__version__)
print(device)

2.6.0.dev20241112
2.5.0.dev20241118
mps


In [4]:
import IPython
import matplotlib.pyplot as plt

In [5]:
symbols = "_-!'(),.:;? abcdefghijklmnopqrstuvwxyz"
look_up = {s: i for i, s in enumerate(symbols)}
symbols = set(symbols)


def text_to_sequence(text):
    text = text.lower()
    return [look_up[s] for s in text if s in symbols]


text = "Hello world! Text to speech!"
print(text_to_sequence(text))

[19, 16, 23, 23, 26, 11, 34, 26, 29, 23, 15, 2, 11, 31, 16, 35, 31, 11, 31, 26, 11, 30, 27, 16, 16, 14, 19, 2]


In [6]:
processor = torchaudio.pipelines.TACOTRON2_WAVERNN_CHAR_LJSPEECH.get_text_processor()

text = "Hello world! Text to speech!"
processed, lengths = processor(text)

print(processed)
print(lengths)

tensor([[19, 16, 23, 23, 26, 11, 34, 26, 29, 23, 15,  2, 11, 31, 16, 35, 31, 11,
         31, 26, 11, 30, 27, 16, 16, 14, 19,  2]])
tensor([28], dtype=torch.int32)


In [9]:
torch.tensor(text_to_sequence(text)) == torch.tensor(processed).squeeze()

/var/folders/x8/23vplcnn1bj6k5l8l0cvk9l80000gn/T/ipykernel_22934/990149607.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(text_to_sequence(text)) == torch.tensor(processed).squeeze()


tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True])

In [10]:
print([processor.tokens[i] for i in processed[0, : lengths[0]]])

['h', 'e', 'l', 'l', 'o', ' ', 'w', 'o', 'r', 'l', 'd', '!', ' ', 't', 'e', 'x', 't', ' ', 't', 'o', ' ', 's', 'p', 'e', 'e', 'c', 'h', '!']


In [1]:
import torch
import torchaudio

# Now try the original code
bundle = torchaudio.pipelines.TACOTRON2_WAVERNN_PHONE_LJSPEECH
processor = bundle.get_text_processor()
text = "Hello world! Text to speech!"

with torch.inference_mode():
    processed, lengths = processor(text)
    print(processed)
    print(lengths)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL dp.preprocessing.text.Preprocessor was not an allowed global by default. Please use `torch.serialization.add_safe_globals([Preprocessor])` or the `torch.serialization.safe_globals([Preprocessor])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

In [2]:
import torch
import torchaudio
from torch.serialization import safe_globals
from dp.preprocessing.text import Preprocessor  # This should work now that you've installed deep-phonemizer

# Use the safe_globals context manager with the actual Preprocessor class
with safe_globals([Preprocessor]):
    bundle = torchaudio.pipelines.TACOTRON2_WAVERNN_PHONE_LJSPEECH
    processor = bundle.get_text_processor()
    
    text = "Hello world! Text to speech!"
    with torch.inference_mode():
        processed, lengths = processor(text)
        print(processed)
        print(lengths)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL dp.preprocessing.text.LanguageTokenizer was not an allowed global by default. Please use `torch.serialization.add_safe_globals([LanguageTokenizer])` or the `torch.serialization.safe_globals([LanguageTokenizer])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.

In [3]:
import torch
import torchaudio
import os
import tempfile

# Get the original loading function to patch
import torchaudio.pipelines._tts.utils as tts_utils
original_load_phonemizer = tts_utils._load_phonemizer

# Create a patched version that forces weights_only=True
def patched_load_phonemizer(file, dl_kwargs=None):
    url = f"https://download.pytorch.org/torchaudio/models/tacotron2/phonemizers/{file}"
    with tempfile.TemporaryDirectory() as tmpdir:
        path = os.path.join(tmpdir, file)
        dl_kwargs = {} if dl_kwargs is None else dl_kwargs
        tts_utils.download_url_to_file(url, path, **dl_kwargs)
        # Use weights_only=True to bypass the unpickling error
        checkpoint = torch.load(path, weights_only=True)
        return checkpoint

# Apply the patch
tts_utils._load_phonemizer = patched_load_phonemizer

# Now try the code again
try:
    bundle = torchaudio.pipelines.TACOTRON2_WAVERNN_PHONE_LJSPEECH
    processor = bundle.get_text_processor()
    
    text = "Hello world! Text to speech!"
    with torch.inference_mode():
        processed, lengths = processor(text)
        print(processed)
        print(lengths)
finally:
    # Restore the original function
    tts_utils._load_phonemizer = original_load_phonemizer

HTTPError: HTTP Error 403: Forbidden

In [1]:
import torch
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import soundfile as sf

/opt/anaconda3/envs/tf/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load models and processor
processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
model.to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
vocoder.to(device)

# Get speaker embedding
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)
speaker_embeddings = speaker_embeddings.to(device)

In [9]:

text = "Hello world this is a new era!"
inputs = processor(text=text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()} 


# Generate speech
with torch.no_grad():
    speech = model.generate_speech(inputs["input_ids"], speaker_embeddings, vocoder=vocoder)
     # If speech is on GPU, move it to CPU for saving
    if speech.device.type == "mps" or speech.device.type == "cuda":
        speech = speech.cpu()


# Save the audio to a file
sf.write("speech.wav", speech.numpy(), samplerate=16000)

print("Speech generated and saved to speech.wav")

Speech generated and saved to speech.wav


In [10]:
from IPython.display import Audio
Audio(speech.numpy(), rate=16000)

In [12]:
import sounddevice as sd

In [13]:
sd.play(speech.numpy(), samplerate=16000)
sd.wait()